In [288]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [289]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.append(str(root / "src" / "tabular"))

In [290]:
import random
from gridworld import STATES, TERMINALS, ACTIONS, reset, sample_step, show

s = reset()
for _ in range(10):
    a = random.randrange(4)
    ns, r, done = sample_step(s, a)
    print(f"{s} --{ACTIONS[a]}-- {ns}  r={r}  done={done}")
    s = ns
    if done:
        break


(2, 0) --^-- (2, 0)  r=-0.04  done=False
(2, 0) --v-- (2, 0)  r=-0.04  done=False
(2, 0) -->-- (2, 1)  r=-0.04  done=False
(2, 1) --v-- (2, 1)  r=-0.04  done=False
(2, 1) -->-- (2, 2)  r=-0.04  done=False
(2, 2) --v-- (2, 2)  r=-0.04  done=False
(2, 2) -->-- (2, 3)  r=-0.04  done=False
(2, 3) -->-- (2, 3)  r=-0.04  done=False
(2, 3) --^-- (2, 3)  r=-0.04  done=False
(2, 3) --^-- (1, 3)  r=-1.0  done=True


In [291]:
from common import value_iteration, greedy_policy

V = value_iteration()
pi = greedy_policy(V)

converged in 19 sweeps


In [292]:
s = reset()
for _ in range(10):
    a = pi[s]
    ns, r, done = sample_step(s, a)
    print(f"{s} --{ACTIONS[a]}-- {ns}  r={r}  done={done}")
    s = ns
    if done:
        break

(2, 0) --^-- (1, 0)  r=-0.04  done=False
(1, 0) --^-- (0, 0)  r=-0.04  done=False
(0, 0) -->-- (0, 1)  r=-0.04  done=False
(0, 1) -->-- (0, 1)  r=-0.04  done=False
(0, 1) -->-- (0, 2)  r=-0.04  done=False
(0, 2) -->-- (0, 3)  r=1.0  done=True


In [293]:
def generate_episode(pi, max_steps=100):
    """[(state, action, reward), ...] — the agent's entire experience of one run."""
    episode = []
    s = reset(explore=True)
    for _ in range(max_steps):
        a = pi(s)
        ns, r, done = sample_step(s, a)
        episode.append((s, a, r))
        s = ns
        if done:
            break
    return episode


In [294]:
from collections import defaultdict


def mc_prediction(pi, n_episodes=5000, gamma=1.0):
    V_est = defaultdict(float)
    counts = defaultdict(int)
    for _ in range(n_episodes):
        episode = generate_episode(lambda s : pi[s])
        G = 0.0
        first_G = {}
        for s, a, r in reversed(episode):
            G = r + gamma * G
            first_G[s] = G
        for s, g in first_G.items():
            counts[s] += 1
            V_est[s] += (g - V_est[s]) / counts[s] 
    return V_est, counts


V_mc, counts = mc_prediction(pi, n_episodes=5000)
show(V=V_mc)
show(V=V)
print({s: counts[s] for s in STATES if s not in TERMINALS})

+0.86  +0.91  +0.96  +1.00 
+0.81    ##   +0.73  -1.00 
+0.75  +0.70  +0.65  +0.38 

+0.85  +0.91  +0.96  +1.00 
+0.80    ##   +0.70  -1.00 
+0.75  +0.70  +0.65  +0.43 

{(0, 0): 3111, (0, 1): 3669, (0, 2): 4855, (1, 0): 2614, (1, 2): 1136, (2, 0): 1982, (2, 1): 1506, (2, 2): 1001, (2, 3): 535}


In [295]:
def epsilon_greedy(Q, s, eps=0.1):
    if random.random() < eps:
        return random.randrange(4)
    return max(range(4), key=lambda a: Q[(s, a)])


def mc_control(n_episodes=50_000, gamma=1.0, eps=0.1):
    Q = defaultdict(float)
    counts = defaultdict(int)
    for _ in range(n_episodes):
        episode = generate_episode(lambda s: epsilon_greedy(Q, s, eps))
        G = 0.0
        first_G = {}
        for s, a, r in reversed(episode):
            G = r + gamma * G
            first_G[(s, a)] = G          # note: keyed by (state, action) now
        for sa, g in first_G.items():
            counts[sa] += 1
            Q[sa] += (g - Q[sa]) / counts[sa]
    return Q, counts


Q, counts = mc_control(eps=0.1)

pi_mc = {
    s: max(range(4), key=lambda a: Q[(s, a)]) for s in STATES if s not in TERMINALS
}
V_mc2 = {s: max(Q[(s, a)] for a in range(4)) for s in STATES if s not in TERMINALS}

show(pi=pi_mc)
show(pi=pi)
show(V=V_mc2)
show(V=V)

  >      >      >    +1.00 
  ^      ##     ^    -1.00 
  ^      <      ^      <   

  >      >      >    +1.00 
  ^      ##     ^    -1.00 
  ^      <      <      <   

+0.82  +0.88  +0.95  +1.00 
+0.77    ##   +0.68  -1.00 
+0.70  +0.65  +0.57  +0.36 

+0.85  +0.91  +0.96  +1.00 
+0.80    ##   +0.70  -1.00 
+0.75  +0.70  +0.65  +0.43 



In [296]:
Q, counts = mc_control(eps=0.01)

pi_mc = {
    s: max(range(4), key=lambda a: Q[(s, a)]) for s in STATES if s not in TERMINALS
}
V_mc2 = {s: max(Q[(s, a)] for a in range(4)) for s in STATES if s not in TERMINALS}

show(pi=pi_mc)
show(pi=pi)
show(V=V_mc2)
show(V=V)

  >      >      >    +1.00 
  ^      ##     ^    -1.00 
  ^      <      <      <   

  >      >      >    +1.00 
  ^      ##     ^    -1.00 
  ^      <      <      <   

+0.84  +0.89  +0.95  +1.00 
+0.79    ##   +0.69  -1.00 
+0.73  +0.68  +0.62  +0.41 

+0.85  +0.91  +0.96  +1.00 
+0.80    ##   +0.70  -1.00 
+0.75  +0.70  +0.65  +0.43 



In [297]:
print({(s, a): counts[(s, a)] for s in [(2, 3), (2, 2)] for a in range(4)})

{((2, 3), 0): 807, ((2, 3), 1): 10, ((2, 3), 2): 15, ((2, 3), 3): 4759, ((2, 2), 0): 35, ((2, 2), 1): 32, ((2, 2), 2): 23, ((2, 2), 3): 10768}
